In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Correlation across all genes

In [2]:
# Configuration and paths
mac = 20
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

abs_phenotypes = True

# Load annotation configuration
if abs_phenotypes:
    config_path = "/home/dnanexus/ukbgym/config_wgs.yaml" # If abs_pheno then deleteriousness direction
else:    
    config_path = "/home/dnanexus/ukbgym/config_olink.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""plof""","""loftee_hc""","""#E31A1C""","""LOFTEE HC""",1
"""plof_consequences""","""consequence_frameshift_variant""","""#E31A1C""","""VEP Frameshift""",1
"""plof_consequences""","""consequence_stop_gained""","""#FB9A99""","""VEP Stop Gained""",1
"""plof_consequences""","""consequence_splice_donor_varia…","""#6A1B9A""","""VEP Splice Donor""",1
"""plof_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
…,…,…,…,…
"""vep_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
"""vep_consequences""","""consequence_start_lost""","""#E65100""","""VEP Start Lost""",1
"""vep_consequences""","""consequence_stop_lost""","""#FFB300""","""VEP Stop Lost""",1


In [3]:
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205.parquet

# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/annotation_files/genebass352genes_olink371genes_annotated_251205_fillna.parquet -o /home/dnanexus/data_dir/genebass352genes_olink371genes_annotated_251205_fillna.parquet

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet -o /home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet

Error: path "/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet"
already exists but -f/--overwrite was not set


In [4]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet")

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # Choose CDS
        (pl.col('vep_cds_relaxed')==True) &
        # ((pl.col('vep_cds_relaxed')==True) | (pl.col('mane_cds')==True)) &
        # (pl.col('non_mane_cds')==False) &

        # Choose non CDS only
        # ((pl.col('vep_cds_relaxed')==False) & (pl.col('mane_cds')==False) & (pl.col('non_mane_cds')==False)) &

        # Choose Gene Body
        # ((pl.col('consequence_upstream_gene_variant') == False) & (pl.col('consequence_downstream_gene_variant') == False)) &

        # Choose VEP consequence
        # (pl.col('consequence_missense_variant') == True) &
        # (pl.col('consequence_synonymous_variant') == True) &
        # (pl.col('consequence_5_prime_utr_variant') == True) &
        # (pl.col('consequence_upstream_gene_variant') == True) &
        # (pl.col('consequence_downstream_gene_variant') == True) &
        # (pl.col('consequence_intron_variant') == True) &

        # Choose MobiDB region
        # (pl.col('mobi_lip_full') == True) &
        # (pl.col('mobi_disorder_full') == True) &

        # Custom variant class filter
        # filter_expression &

        # Choose regulatory region
        # (pl.col('encode_eh_pr') == True) &
        # (pl.col('encode_all_tf') == True) &
        
        # Proximity to TSS
        # (pl.col('dist_to_tss') >= min_range) &
        # (pl.col('dist_to_tss') <= max_range) &

        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
    # .with_columns(
    #     core_promoter = pl.col('dist_to_tss').abs() <= 50,
    #     encode_annotated = pl.col('not_in_encode') == False,
    #     encode_eh_pr = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_pls', 'encode_pels', 'encode_dels']),
    #     encode_all_tf = pl.any_horizontal((pl.col(c) == 1) for c in ['encode_tf', 'encode_ca_tf']),
    # )
)

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

anno = (
    anno
    .select(
        # set(['id', 'region', 'tss', 'strand', 'gene_length', 'gene_name', 'dist_to_tss']).union(set(existing_annos))
        set(['id', 'region']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

consequence_frameshift_variant,five_prime_utr_variant_consequence_ustop_gained,verphylop,five_prime_utr_variant_consequence_uaug_gained,delta_score,abexp_abs_max,cadd_raw,consequence_stop_lost,consequence_synonymous_variant,pangolin_score,consequence_start_lost,loftee_hc,consequence_splice_acceptor_variant,absplice_dna_max,five_prime_utr_variant_consequence_uaug_lost,esmscoremissense,loftee_lc,low_complexity_domain,score_pai3d,am_pathogenicity,absplice2_max,mobi_curated_disorder_priority,gpn_score,mobi_lip_full,consequence_missense_variant,id,consequence_stop_gained,promoterai,ted_domain,consequence_splice_donor_variant,region,five_prime_utr_variant_consequence_ustop_lost
i8,u8,f32,u8,f32,f32,f32,i8,i8,f32,i8,i8,i8,f32,u8,f32,i8,bool,f32,f32,f32,bool,f32,bool,i8,str,i8,f32,bool,i8,str,u8
0,0,8.687,0,0.05,0.060269,3.681756,0,0,0.03,0,0,0,0.003,0,-4.246,0,false,0.699444,0.2063,0.000327,false,-11.54,false,1,"""chr5:87376465:A:T""",0,0.0,false,0,"""ENSG00000145715""",0
0,0,2.372,0,0.0,0.946601,7.69086,0,0,0.06,0,1,0,0.003,0,0.0,0,false,0.0,0.0,0.002165,false,-1.6,true,0,"""chr1:177933615:G:A""",1,0.0,false,0,"""ENSG00000120341""",0
0,0,-1.5,0,0.0,0.012463,0.528766,0,1,0.0,0,0,0,0.001,0,0.0,0,false,0.0,0.0,0.000033,false,0.36,false,0,"""chr2:96816550:C:T""",0,-0.143,false,0,"""ENSG00000168763""",0
0,0,6.948,0,0.08,0.015385,3.032575,0,0,0.03,0,0,0,0.0,0,-6.253,0,false,0.0,0.0,0.000034,false,-8.09,true,1,"""chr2:178598858:G:A""",0,0.0,false,0,"""ENSG00000155657""",0
0,0,-0.102,0,0.0,0.160049,4.428444,0,0,0.0,0,0,0,0.003,0,-5.721,0,false,0.550982,0.2002,0.000321,false,-5.12,false,1,"""chr2:128318064:C:T""",0,0.0,true,0,"""ENSG00000136720""",0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,-0.6,0,0.01,1.39091,5.248261,0,0,0.03,0,1,0,0.003,0,0.0,0,false,0.0,0.0,0.002172,false,-6.74,false,0,"""chr4:15816634:C:A""",1,0.0032,true,0,"""ENSG00000004468""",0
0,0,0.709,0,0.0,0.013111,1.2186,0,1,0.0,0,0,0,0.001,0,0.0,0,true,0.0,0.0,0.000033,false,-4.14,false,0,"""chr15:78077599:C:A""",0,-0.0234,false,0,"""ENSG00000167202""",0
0,0,2.934,0,0.0,0.006189,0.213689,0,1,0.0,0,0,0,0.001,0,0.0,0,false,0.0,0.0,0.000033,false,-0.88,true,0,"""chr14:67783209:G:A""",0,0.0,true,0,"""ENSG00000072121""",0


In [5]:
selected_annos = anno_config_df.filter(
    (pl.col('annotation') == 'loftee_hc') # LOFTEE
)['annotation'].to_list()

melted_anno = (
    anno
    
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    ).with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
)
melted_anno

id,region,annotation,annotation_score
str,str,str,f32
"""chr5:87376465:A:T""","""ENSG00000145715""","""loftee_hc""",0.0
"""chr1:177933615:G:A""","""ENSG00000120341""","""loftee_hc""",1.0
"""chr2:96816550:C:T""","""ENSG00000168763""","""loftee_hc""",0.0
"""chr2:178598858:G:A""","""ENSG00000155657""","""loftee_hc""",0.0
"""chr2:128318064:C:T""","""ENSG00000136720""","""loftee_hc""",0.0
…,…,…,…
"""chr4:15816634:C:A""","""ENSG00000004468""","""loftee_hc""",1.0
"""chr15:78077599:C:A""","""ENSG00000167202""","""loftee_hc""",0.0
"""chr14:67783209:G:A""","""ENSG00000072121""","""loftee_hc""",0.0


In [6]:
# Subset Olink RVAT significant

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/blacklist/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet -o /home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet

olink_whitelist = (
    pl.read_parquet('/home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burden_regression_results.parquet')
    .rename({'gene': 'region'})
    .filter(pl.col('padj_perm')<=0.05)
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

olink_whitelist

Error: path "/home/dnanexus/data_dir/olink/proteomics_prs_am_loftee_mac20_burd
en_regression_results.parquet" already exists but -f/--overwrite was not set


region,phenotype
str,str
"""ENSG00000213066""","""ENSG00000213066_olink"""
"""ENSG00000107201""","""ENSG00000107201_olink"""
"""ENSG00000113739""","""ENSG00000113739_olink"""
"""ENSG00000085514""","""ENSG00000085514_olink"""
"""ENSG00000071051""","""ENSG00000071051_olink"""
…,…
"""ENSG00000092529""","""ENSG00000092529_olink"""
"""ENSG00000145730""","""ENSG00000145730_olink"""
"""ENSG00000135218""","""ENSG00000135218_olink"""


In [7]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/olink_all_genes_EURunrelated_appv_percentiles.parquet -o /home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet

# Read Olink phenotype data
olink_appv = pl.scan_parquet("/home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet")

# Merge phenotype data and annotation data
gp_corr_df = (
    olink_appv
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    # .join(
    #     olink_whitelist.lazy(),
    #     on=["region", "phenotype"],
    #     how="semi"
    # )
    .with_columns(
        mean_pheno_value = pl.when(abs_phenotypes)
        .then(pl.col('mean_pheno_value').abs())
        .otherwise(pl.col('mean_pheno_value'))
    )
    .join(
        melted_anno.lazy(), 
        on="id", 
        how="inner"
    )
    .filter(
        pl.col('region') + '_olink' == pl.col('phenotype')
    )
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['mean_pheno_value', 'annotation_score']
    )
    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True)
    )
    .rename({
        'phenotype': 'olink_prot',
    })
    .collect(engine='streaming')
)

gp_corr_df

Error: path
"/home/dnanexus/olink_all_genes_EURunrelated_appv_percentiles.parquet" already
exists but -f/--overwrite was not set


region,olink_prot,annotation,n_variants,correlation
str,str,str,u64,f64
"""ENSG00000179639""","""ENSG00000179639_olink""","""loftee_hc""",68,-0.090253
"""ENSG00000099937""","""ENSG00000099937_olink""","""loftee_hc""",117,0.261824
"""ENSG00000169379""","""ENSG00000169379_olink""","""loftee_hc""",100,0.30137
"""ENSG00000155629""","""ENSG00000155629_olink""","""loftee_hc""",196,NaN
"""ENSG00000110400""","""ENSG00000110400_olink""","""loftee_hc""",188,-0.010106
…,…,…,…,…
"""ENSG00000173535""","""ENSG00000173535_olink""","""loftee_hc""",64,0.187554
"""ENSG00000003056""","""ENSG00000003056_olink""","""loftee_hc""",55,0.183556
"""ENSG00000143167""","""ENSG00000143167_olink""","""loftee_hc""",105,0.096846


In [ ]:
gp_corr_df = (
    gp_corr_df
    .select(['region', 'olink_prot', 'annotation', 'n_variants', 'correlation'])
    .join(
        anno_config_df,
        on='annotation'
    )
    .with_columns(
        corr_beta = pl.col('correlation')*pl.col('annotation_dir')
    )
)

gp_corr_df

region,olink_prot,annotation,n_variants,correlation,category,color,label,annotation_dir,corr_beta
str,str,str,u64,f64,str,str,str,i8,f64
"""ENSG00000133110""","""ENSG00000133110_olink""","""loftee_hc""",180,0.131346,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.131346
"""ENSG00000164619""","""ENSG00000164619_olink""","""loftee_hc""",136,0.253127,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.253127
"""ENSG00000092529""","""ENSG00000092529_olink""","""loftee_hc""",228,0.022074,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.022074
"""ENSG00000186567""","""ENSG00000186567_olink""","""loftee_hc""",157,0.098988,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.098988
"""ENSG00000004455""","""ENSG00000004455_olink""","""loftee_hc""",59,0.177346,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.177346
…,…,…,…,…,…,…,…,…,…
"""ENSG00000145703""","""ENSG00000145703_olink""","""loftee_hc""",379,0.287201,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.287201
"""ENSG00000179954""","""ENSG00000179954_olink""","""loftee_hc""",409,0.271907,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.271907
"""ENSG00000105492""","""ENSG00000105492_olink""","""loftee_hc""",101,0.154031,"""plof""","""#E31A1C""","""LOFTEE HC""",1,0.154031


In [8]:
gp_corr_df.filter(pl.col('annotation')=='loftee_hc').write_parquet('/home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet')

In [9]:
!dx upload /home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/olink_all_mac20_lofteeHC_correlations.parquet

[===========================================================>] Uploaded 41,918 of 41,918 bytes (100%) /home/dnanexus/data_dir/olink_all_mac20_lofteeHC_correlations.parquet
ID                                file-J5jVGk0Jg0y0FxfpYxbjyj7y
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/REGENIE_results
Name                              olink_all_mac20_lofteeHC_correlations.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Wed Jan 21 18:09:40 2026
Created by                        shubhankar
 via the job                      job-J5jKzb8Jg0yGx8PvvjFky1Xy
Last modified                     Wed Jan 21 18:09:40 2026
Media type                        
archivalState          